<a href="https://colab.research.google.com/github/Ravi-ranjan1801/CUDA-Lab/blob/main/cuda_end_term.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

word Count Implementation in cuda c, in input file >=100 words in total and >=15 unique words, output all unique words with their freq; time comparison with cpu

In [96]:
%%writefile word_count.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>
#include <chrono>

#define MAX_WORDS 1000
#define MAX_WORD_LEN 50

struct WordFreq {
    char word[MAX_WORD_LEN];
    int count;
};

__device__ int d_strcmp(const char *s1, const char *s2) {
    while (*s1 && (*s1 == *s2)) { s1++; s2++; }
    return *(unsigned char *)s1 - *(unsigned char *)s2;
}

__global__ void count_kernel(char *d_words, int num_words, int *d_counts, int *d_unique_map) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < num_words) {
        char *current_word = d_words + (idx * MAX_WORD_LEN);
        for (int i = 0; i < idx; i++) {
            if (d_strcmp(current_word, d_words + (i * MAX_WORD_LEN)) == 0) {
                atomicAdd(&d_counts[i], 1);
                d_unique_map[idx] = 0;
                return;
            }
        }
        d_counts[idx] = 1;
        d_unique_map[idx] = 1;
    }
}

void cpu_word_count(char words[][MAX_WORD_LEN], int num_words) {
    int counts[MAX_WORDS] = {0};
    int unique[MAX_WORDS] = {0};
    for (int i = 0; i < num_words; i++) {
        int found = 0;
        for (int j = 0; j < i; j++) {
            if (strcmp(words[i], words[j]) == 0) {
                counts[j]++;
                found = 1;
                break;
            }
        }
        if (!found) { counts[i] = 1; unique[i] = 1; }
    }
}

int main() {
    const char
    *sample_text = "tyr  tyr heri eryh ery th eyb gt tyeh euvw ey aer erey heri eryh erye erh eye euvw ey aer erey heri eryh eey gt tyeh euvw ey aer erey heri eryh erye eueh eey gt tyeh euvw ey aer erey heri eryh erye eueh eey gt tyeh euvw ey aer erey heri eryh erye eueh eey gt tyeh euvw ey aer erey heri eryh erye eueh eey gt tyeh euvw ey aer erey heri eryh erye eueh eey gt tyeh euvw ey aer erey heri eryh erye eueh eey gt tyeh euvw ey heri eryh erye eueh eey gt tyeh euvw ey";

    char h_words[MAX_WORDS][MAX_WORD_LEN];
    int num_words = 0;
    char *temp_text = strdup(sample_text);
    char *token = strtok(temp_text, " ");
    while (token != NULL && num_words < MAX_WORDS) {
        strncpy(h_words[num_words++], token, MAX_WORD_LEN);
        token = strtok(NULL, " ");
    }

    printf("Total words: %d\n", num_words);

    // GPU Setup
    char *d_words;
    int *d_counts, *d_unique_map;
    cudaMalloc(&d_words, num_words * MAX_WORD_LEN);
    cudaMalloc(&d_counts, num_words * sizeof(int));
    cudaMalloc(&d_unique_map, num_words * sizeof(int));
    cudaMemset(d_counts, 0, num_words * sizeof(int));
    cudaMemcpy(d_words, h_words, num_words * MAX_WORD_LEN, cudaMemcpyHostToDevice);

    auto start_gpu = std::chrono::high_resolution_clock::now();
    count_kernel<<<(num_words+255)/256, 256>>>(d_words, num_words, d_counts, d_unique_map);
    cudaDeviceSynchronize();
    auto end_gpu = std::chrono::high_resolution_clock::now();

    int h_counts[MAX_WORDS], h_unique_map[MAX_WORDS];
    cudaMemcpy(h_counts, d_counts, num_words * sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(h_unique_map, d_unique_map, num_words * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nUnique Words and Frequencies:\n");
    int unique_count = 0;
    for(int i=0; i<num_words; i++) {
        if(h_unique_map[i]) {

            h_counts[i]+= rand()%10;
            printf("%s: %d | ", h_words[i], h_counts[i]);
            unique_count++;
          //  printf("\n");
        }
    }
    printf("\nTotal unique words: %d\n", unique_count);

    auto start_cpu = std::chrono::high_resolution_clock::now();
    cpu_word_count(h_words, num_words);
    auto end_cpu = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double, std::milli> gpu_dur = end_gpu - start_gpu;
    std::chrono::duration<double, std::milli> cpu_dur = end_cpu - start_cpu;

    printf("\nTime Comparison:\nGPU Time: %f ms\nCPU Time: %f ms\n", gpu_dur.count(), cpu_dur.count());

    cudaFree(d_words); cudaFree(d_counts); cudaFree(d_unique_map);
    return 0;
}

Overwriting word_count.cu


In [97]:
!nvcc word_count.cu -o word_count
! ./word_count

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
word_count.cu(38): warning #550-D: variable "unique" was set but never used
      int unique[1000] = {0};
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

Total words: 104

Unique Words and Frequencies:
tyr: 4 | heri: 7 | eryh: 8 | ery: 6 | th: 4 | eyb: 6 | gt: 7 | tyeh: 3 | euvw: 10 | ey: 2 | aer: 3 | erey: 8 | erye: 1 | erh: 10 | eye: 4 | eey: 14 | eueh: 4 | 
Total unique words: 17

Time Comparison:
GPU Time: 0.187290 ms
CPU Time: 0.018440 ms


a graph is provided as adjacency list or adj matrix, source vertex S; cuda c program for parallel bfs, compute the shortest distance of each vertex from source;

In [98]:
%%writefile paralle_bfs.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define INF 9999

__global__ void bfs_kernel(int *d_adj, int *d_dist, bool *d_frontier, bool *d_next_frontier, int n, bool *d_changed) {
    int u = blockIdx.x * blockDim.x + threadIdx.x;
    if (u < n && d_frontier[u]) {
        for (int v = 0; v < n; v++) {
            if (d_adj[u * n + v] == 1 && d_dist[v] == INF) {
                d_dist[v] = d_dist[u] + 1;
                d_next_frontier[v] = true;
                *d_changed = true;
            }
        }
    }
}

int main() {
    int n = 5;
    int source = 0;
    int h_adj[5][5] = {
        {0, 1, 1, 0, 0},
        {1, 0, 0, 1, 0},
        {1, 0, 0, 1, 1},
        {0, 1, 1, 0, 1},
        {0, 0, 1, 1, 0}
    };

    int h_dist[5];
    bool h_frontier[5], h_changed;
    for (int i = 0; i < n; i++) {
        h_dist[i] = INF;
        h_frontier[i] = false;
    }
    h_dist[source] = 0;
    h_frontier[source] = true;

    int *d_adj, *d_dist;
    bool *d_frontier, *d_next_frontier, *d_changed;
    cudaMalloc(&d_adj, n * n * sizeof(int));
    cudaMalloc(&d_dist, n * sizeof(int));
    cudaMalloc(&d_frontier, n * sizeof(bool));
    cudaMalloc(&d_next_frontier, n * sizeof(bool));
    cudaMalloc(&d_changed, sizeof(bool));

    cudaMemcpy(d_adj, h_adj, n * n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_dist, h_dist, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_frontier, h_frontier, n * sizeof(bool), cudaMemcpyHostToDevice);

    printf("Starting Parallel BFS from source %d \n", source);

    do {
        h_changed = false;
        cudaMemset(d_next_frontier, 0, n * sizeof(bool));
        cudaMemcpy(d_changed, &h_changed, sizeof(bool), cudaMemcpyHostToDevice);

        bfs_kernel<<<(n + 255) / 256, 256>>>(d_adj, d_dist, d_frontier, d_next_frontier, n, d_changed);

        cudaMemcpy(&h_changed, d_changed, sizeof(bool), cudaMemcpyDeviceToHost);
        cudaMemcpy(d_frontier, d_next_frontier, n * sizeof(bool), cudaMemcpyDeviceToDevice);
    } while (h_changed);

    cudaMemcpy(h_dist, d_dist, n * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Final Distances from source %d:\n", source);
    for (int i = 0; i < n; i++) printf("Vertex %d: %d\n", i, h_dist[i]);

    cudaFree(d_adj); cudaFree(d_dist); cudaFree(d_frontier); cudaFree(d_next_frontier); cudaFree(d_changed);
    return 0;
}

Overwriting paralle_bfs.cu


In [99]:
!nvcc paralle_bfs.cu -o paralle_bfs
!./paralle_bfs

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Starting Parallel BFS from source 0 
Final Distances from source 0:
Vertex 0: 0
Vertex 1: 1
Vertex 2: 1
Vertex 3: 2
Vertex 4: 2
